# Comp Lab 3 — Single-Cell RNA-seq Annotation: Markers vs. the Reference Model
**BIO 462/594 Molecular Biology | Thursday, December 3 (last day of class) | OSC OnDemand (Pitzer, high-memory CPU)**

Today you annotate a real single-cell dataset — **44,721 peripheral blood mononuclear cells (PBMCs)** from COVID-19 patients and healthy
donors (Su et al., *Nature Medicine* 2020) — **two different ways**:

1. **Marker-based annotation** — cluster the cells, then use canonical marker genes (what you learned in lecture) to name each cluster. Bottom-up: *the data tells you what to look at.*
2. **CellTypist** — a reference model trained on ~300,000 curated immune cells (Domínguez Conde et al., *Science* 2022). Top-down: *the model tells you what it sees.*

Then you investigate **where they disagree** — because they will disagree, and the disagreements are where the biology (and the model's
blind spots) live.

**Your write-up (30% of the portfolio) is assigned today and due Friday, Dec 11.**

## How this notebook works
- Run top to bottom with **Shift+Enter**. Heavy steps (PCA, clustering) take up to a minute — the printout tells you what finished.
- **Your turn** cells ask for written answers or a small code change.
- The dataset ships with the original authors' own annotations (`cell.type.fine`). **Do not use them to annotate** — they are the answer
key, revealed in Part 6 to check your work. This is also why you should always check what is in `X` vs `raw` before trusting a public
dataset: the stored matrix here is *normalized*, and we will pull the true counts out of `raw`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, scanpy as sc, warnings
warnings.simplefilter('ignore')
plt.rcParams['font.family'] = ['Liberation Sans', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=90, facecolor='white')

adata = sc.read_h5ad('data/su2020_pbmc_covid.h5ad')
print(adata)
print()
print('disease labels:', adata.obs['disease'].value_counts().to_dict())
print()
print('NOTE: the stored matrix is normalized, and genes are Ensembl IDs.')
print('We will rebuild the object with true counts (from raw) and gene symbols.')

---
## Part 1 — QC: keep the good cells (10 min)

The authors already computed QC metrics. We filter cells with unusually many genes (doublets), too few genes (empty droplets), or high
mitochondrial fraction (dying cells).

In [ ]:
import scipy.sparse as sp
# true integer counts live in .raw; gene symbols live in var['feature_name']
counts = sp.csr_matrix(adata.raw.X)
symbols = adata.var['feature_name'].astype(str).values
assert list(adata.raw.var_names) == list(adata.var_names)  # same gene order

import anndata as ad
adata = ad.AnnData(X=counts, obs=adata.obs.copy())
adata.var_names = pd.Index(symbols, name='gene_symbol')
adata.var_names_make_unique()
adata.obs['n_counts'] = np.ravel(adata.X.sum(1))
adata.obs['n_genes'] = np.ravel((adata.X > 0).sum(1))
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
print('before QC:', adata.n_obs, 'cells,', adata.n_vars, 'genes')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ['n_genes', 'n_counts', 'pct_counts_mt']):
    ax.hist(adata.obs[col], bins=60, color='#0279EE')
    ax.set_title(col)
fig.tight_layout()
plt.show()

In [ ]:
adata = adata[(adata.obs['n_genes'] >= 200) & (adata.obs['n_genes'] <= 5000)
              & (adata.obs['pct_counts_mt'] < 30)].copy()
print('after QC:', adata.n_obs, 'cells,', adata.n_vars, 'genes')

---
## Part 2 — Cluster the cells (15 min)

Standard recipe: normalize → log → highly variable genes → scale → PCA → neighbors → Leiden clustering. This takes about a minute —
watch the printouts.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers['lognorm'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=3000, subset=False)
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=30, random_state=0)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.tl.leiden(adata, resolution=0.5, key_added='leiden', random_state=0, flavor='igraph', n_iterations=2, directed=False)
print('clusters:', adata.obs['leiden'].nunique())

In [ ]:
sc.pl.umap(adata, color=['leiden', 'disease'], frameon=False, show=False)
plt.savefig('lab3_umap_overview.png', dpi=140, bbox_inches='tight')
plt.show()

**Your turn 1.** How many Leiden clusters did we get? Do any clusters look disease-specific (mostly COVID or mostly healthy)?

---
## Part 3 — Marker-based annotation (15 min)

You are the expert. Below is a dot plot of canonical PBMC markers per cluster. Dot size = fraction of cells expressing; color = average
expression level.

| cell type | marker genes |
|---|---|
| CD4+ T cell | CD3D, CD3E, IL7R, S100A4 (memory) / CCR7, TCF7 (naive) |
| CD8+ T cell | CD3D, CD8A, CD8B, GZMK (effector-memory) |
| NK cell | NKG7, GNLY, KLRD1 |
| B cell | MS4A1 (CD20), CD79A, TCL1A (naive) |
| Plasmablast | MZB1, JCHAIN, XBP1, IGKC |
| CD14+ monocyte | LST1, S100A8, S100A9, CD14 |
| FCGR3A+ (CD16) monocyte | FCGR3A, MS4A7, LST1 |
| Dendritic cell | FCER1A, CST3, CLEC10A |
| Plasmacytoid DC | LILRA4, GZMB, IRF7 |
| Platelet | PPBP, PF4 |
| Reticulocyte/RBC | HBB, HBA1 |
| Proliferating | MKI67, TOP2A |

In [ ]:
markers = ['CD3D','CD3E','IL7R','S100A4','CCR7','CD8A','GZMK','NKG7','GNLY','KLRD1',
           'MS4A1','CD79A','TCL1A','MZB1','JCHAIN','XBP1','IGKC','CD14','LST1','S100A8',
           'FCGR3A','MS4A7','FCER1A','CLEC10A','LILRA4','IRF7','PPBP','PF4','HBB','HBA1','MKI67']
markers = [m for m in markers if m in adata.var_names]
sc.pl.dotplot(adata, markers, groupby='leiden', show=False)
plt.savefig('lab3_dotplot.png', dpi=140, bbox_inches='tight')
plt.show()

**Your turn 2 (worksheet).** For each Leiden cluster, write your marker-based label. Double-click and edit:

| cluster | my label | evidence (top 2 markers) |
|---|---|---|
| 0 | | |
| 1 | | |
| 2 | | |
| ... | | |

Then encode your labels below and plot them on the UMAP.

In [ ]:
my_labels = {
    '0': 'CD14 Monocyte',   # example row - replace with your calls
    # ... fill in every cluster
}
adata.obs['my_annotation'] = adata.obs['leiden'].map(my_labels)
sc.pl.umap(adata, color='my_annotation', frameon=False, show=False)
plt.savefig('lab3_umap_myannotation.png', dpi=140, bbox_inches='tight')
plt.show()

---
## Part 4 — CellTypist: the reference model (15 min)

CellTypist is a logistic-regression classifier trained on a curated atlas of ~300,000 immune cells. It transfers labels from its reference
onto your data, with a **confidence score** for every cell. Load the `Immune_All_Low` model (fine-grained immune cell types, all donors).

In [ ]:
import celltypist, os
from celltypist import models
models.download_models(model='Immune_All_Low.pkl', force_update=False)
model_path = 'Immune_All_Low.pkl'
if os.path.exists('data/Immune_All_Low.pkl'):
    model_path = 'data/Immune_All_Low.pkl'   # staged with the lab data
tmp = adata.copy()
tmp.X = tmp.layers['lognorm']          # CellTypist wants log1p-normalized X
predictions = celltypist.annotate(tmp, model=model_path,
                                  majority_voting=True, over_clustering='leiden')
predicted = predictions.predicted_labels
adata.obs['celltypist'] = predicted['predicted_labels'].values
adata.obs['celltypist_conf'] = predictions.probability_matrix.max(axis=1).values
adata.obs[['celltypist', 'celltypist_conf']].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.umap(adata, color='celltypist', frameon=False, ax=axes[0], show=False, title='CellTypist labels')
sc.pl.umap(adata, color='celltypist_conf', frameon=False, ax=axes[1], show=False,
           title='CellTypist confidence', cmap='viridis', vmin=0, vmax=1)
plt.savefig('lab3_umap_celltypist.png', dpi=140, bbox_inches='tight')
plt.show()

**Your turn 3.** Where is CellTypist confident (bright) and where is it uncertain (dark)? Do the uncertain islands match clusters where
*you* were unsure in Part 3?

---
## Part 5 — Where do they disagree? (15 min)

Now the main event. Cross-tabulate your marker-based labels against CellTypist's, and hunt for disagreement — especially clusters where
CellTypist is confident but disagrees with your markers, or where it is unsure.

In [ ]:
ct = pd.crosstab(adata.obs['my_annotation'], adata.obs['celltypist'])
ct = ct[ct.sum().sort_values(ascending=False).index]
ct

In [ ]:
# per-cluster: CellTypist's modal label, its agreement with you, and mean confidence
summary = []
for cl, sub in adata.obs.groupby('leiden'):
    modal = sub['celltypist'].value_counts()
    summary.append(dict(cluster=cl, n=len(sub),
                        my_label=my_labels.get(cl, '?'),
                        celltypist_modal=modal.index[0],
                        celltypist_frac=round(modal.iloc[0]/len(sub), 2),
                        mean_conf=round(sub['celltypist_conf'].mean(), 2),
                        agrees=my_labels.get(cl, '?') == modal.index[0]))
summary = pd.DataFrame(summary)
summary

**Your turn 4 (the write-up core).** Pick the two most interesting disagreements from the table above and adjudicate them with marker
genes. For each: what did you call it, what did CellTypist call it, and which is better supported? Useful plots below — change the genes
to fit your case.

In [ ]:
# example adjudication plots - replace genes to fit YOUR disagreement cases
sc.pl.umap(adata, color=['ISG15', 'IFIT1', 'FCGR3A', 'MZB1'], frameon=False, show=False)
plt.savefig('lab3_adjudication.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# sub-cluster a disputed cluster at higher resolution (change the cluster id)
disputed = adata.obs['leiden'] == '0'    # <-- change me
tmp = adata[disputed].copy()
sc.tl.leiden(tmp, resolution=1.0, key_added='sub', random_state=0, flavor='igraph', n_iterations=2, directed=False)
sc.pl.dotplot(tmp, ['CD14','LST1','S100A8','FCGR3A','MS4A7','ISG15'], groupby='sub', show=False)
plt.show()

---
## Part 6 — Reveal: the authors' annotations (5 min)

The Su et al. authors annotated every cell by hand (`cell.type.fine`). Compare all three side by side.

In [ ]:
sc.pl.umap(adata, color=['my_annotation', 'celltypist', 'cell.type.fine'],
           frameon=False, show=False)
plt.savefig('lab3_umap_reveal.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# NOTE: exact string matching is meaningless here - the authors' vocabulary
# ('CD14 Monocyte') differs from CellTypist's ('CD14+ Monocytes'). Compare the
# VIEWS instead: each row of this table is one of the authors' types, and the
# columns show what you and CellTypist called those same cells.
ct_reveal = pd.crosstab(adata.obs['cell.type.fine'],
                        [adata.obs['my_annotation'].astype(str).replace('nan', 'unlabeled'),
                         adata.obs['celltypist']])
ct_reveal

**Discussion (whole class):**
1. Which disagreements were real biology (a state the reference model had never seen — e.g. IFN-stimulated cells in COVID) and which were annotation-granularity mismatches (e.g. "CD4m T" vs "T helper cells")?
2. CellTypist's reference is dominated by healthy adult blood. What happens to its confidence on disease tissue? What does that imply for using reference models on new biology?
3. The authors' labels are not ground truth either — they are expert curation. When a reference model, your markers, AND the original authors all disagree, what would you actually trust?

**Lab 3 write-up — due Friday, Dec 11 (30% of portfolio).** See the write-up handout. The 1-page final reflection (10%) is also due that day.